# Research Style Writer (RSW)
## Complete Pipeline Walkthrough

This notebook walks through every stage of the pipeline for generating human-like academic research papers.

**Pipeline Stages:**
1. Configuration & Environment Setup
2. PDF Ingestion & Text Extraction
3. Text Chunking
4. Writing Style Analysis
5. Web Search (Tavily API)
6. Prompt Construction
7. LLM-Based Section Generation
8. Automated Evaluation & Iterative Refinement
9. Multi-Pass Humanization Pipeline
10. Academic Image Search & Insertion
11. PDF / DOCX Export
12. Full End-to-End Paper Generation

**Prerequisites:** Python 3.10+, installed dependencies (`pip install -r requirements.txt`), `.env` file with API keys.

**Project:** Research Style Writer (RSW) — Mini Project

In [ ]:
import sys, os

# Set project root (adjust if notebook is in a different location)
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.getcwd())) if 'deliverables' in os.getcwd() else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

---
## Step 1: Configuration & Environment Setup

The `Settings` dataclass (in `src/core/config.py`) reads all API keys and parameters from a `.env` file using `python-dotenv`. It supports multiple LLM providers (xAI/Grok, Groq, Google Gemini, OpenRouter) and validates that the selected provider has a valid API key.

In [ ]:
from src.core.config import settings

print(f"LLM Provider       : {settings.llm_provider}")
print(f"Embedding Provider  : {settings.embedding_provider}")
print(f"Embedding Model     : {settings.embedding_model_name}")
print(f"Chunk Size          : {settings.chunk_size}")
print(f"Chunk Overlap       : {settings.chunk_overlap}")
print(f"Top-K Retrieval     : {settings.top_k}")
print(f"Min/Max Ref Files   : {settings.min_files} - {settings.max_files}")

try:
    settings.validate()
    print("\nConfiguration validated successfully.")
except ValueError as e:
    print(f"\nConfiguration error: {e}")

---
## Step 2: PDF Ingestion & Text Extraction

The PDF loader (`src/core/ingestion/pdf_loader.py`) uses **PyMuPDF (fitz)** to extract text page by page. If a page has corrupted fonts (detected by high `?` character ratio), it falls back to **Gemini Vision (VLM)** for OCR-based extraction.

**Output:** A list of `{text, source, page}` dictionaries — one per page.

In [ ]:
from src.core.ingestion.pdf_loader import extract_text_from_pdf

sample_pdf_dir = os.path.join(PROJECT_ROOT, "data", "uploads")
sample_pdfs = [f for f in os.listdir(sample_pdf_dir) if f.endswith(".pdf")] if os.path.isdir(sample_pdf_dir) else []

if sample_pdfs:
    with open(os.path.join(sample_pdf_dir, sample_pdfs[0]), "rb") as f:
        pages = extract_text_from_pdf(f.read(), sample_pdfs[0])
    print(f"Extracted {len(pages)} pages from '{sample_pdfs[0]}'")
    if pages:
        print(f"Page 1 preview ({len(pages[0]['text'].split())} words):")
        print(pages[0]['text'][:250] + "...")
else:
    # Synthetic demo data
    pages = [
        {"text": "Machine learning has transformed natural language processing. "
                 "Recent advances in transformer architectures have enabled "
                 "unprecedented capabilities in text generation and understanding. "
                 "This paper examines the implications of these developments.",
         "source": "demo_paper.pdf", "page": 1},
        {"text": "The methodology employs a multi-stage pipeline combining "
                 "retrieval-augmented generation with style-aware text synthesis.",
         "source": "demo_paper.pdf", "page": 2},
    ]
    print(f"Using {len(pages)} synthetic demo pages.")

---
## Step 3: Text Chunking

Uses LangChain's `RecursiveCharacterTextSplitter` to break extracted pages into overlapping chunks. This ensures style analysis captures patterns across paragraph boundaries.

- **Chunk size:** 1000 characters (configurable)
- **Overlap:** 200 characters

In [ ]:
from src.core.ingestion.chunker import chunk_pages

chunks = chunk_pages(pages)
print(f"Input pages  : {len(pages)}")
print(f"Output chunks: {len(chunks)}")
print(f"Chunk size   : {settings.chunk_size} chars, overlap: {settings.chunk_overlap} chars")

if chunks:
    print(f"\nFirst chunk:")
    print(f"  Source: {chunks[0]['source']}, Page: {chunks[0]['page']}")
    print(f"  Text: {chunks[0]['text'][:200]}...")

---
## Step 4: Writing Style Analysis

The `StyleProfile` dataclass (`src/core/style/style_analyzer.py`) extracts **12 quantitative features** from the reference papers:

| Feature | Description |
|---------|-------------|
| `avg_sentence_length` | Mean words per sentence |
| `sentence_length_std` | Standard deviation of sentence lengths |
| `lexical_diversity` | Type-token ratio (unique/total words) |
| `top_transition_words` | Most used hedging/transition words |
| `passive_voice_ratio` | Fraction of passive voice sentences |
| `avg_paper_word_count` | Average word count per paper |
| `avg_paragraph_length` | Mean words per paragraph |
| `paragraph_length_std` | Paragraph length variation |
| `sentence_initial_variety` | % of sentences NOT starting with article/pronoun |
| `questions_per_1000_words` | Rhetorical question frequency |
| `first_person_ratio` | Usage of we/our/us |
| `avg_paragraphs_per_section` | Structural density |

These features are converted to natural language instructions via `to_prompt_text()` and fed to the LLM.

In [ ]:
from src.core.style.style_analyzer import analyze_style

style_profile = analyze_style(chunks)

print(f"Avg sentence length    : {style_profile.avg_sentence_length:.1f} words")
print(f"Sentence length StdDev : {style_profile.sentence_length_std:.1f}")
print(f"Lexical diversity      : {style_profile.lexical_diversity:.3f}")
print(f"Passive voice ratio    : {style_profile.passive_voice_ratio:.2f}")
print(f"Avg paper word count   : {style_profile.avg_paper_word_count}")
print(f"Avg paragraph length   : {style_profile.avg_paragraph_length:.0f} words")
print(f"Sentence-initial variety: {style_profile.sentence_initial_variety:.3f}")
print(f"First-person ratio     : {style_profile.first_person_ratio:.4f}")
print(f"Questions per 1000w    : {style_profile.questions_per_1000_words:.2f}")

print(f"\nTop transition words:")
for word, count in style_profile.top_transition_words[:5]:
    print(f"  {word}: {count}")

print(f"\nStyle prompt text (fed to LLM):")
print(style_profile.to_prompt_text())

---
## Step 5: Web Search (Tavily API)

Each section gets its own web search via the **Tavily API** to gather real, current domain content. This grounds the LLM output in factual sources rather than relying solely on parametric knowledge.

- Returns: title, URL, and extracted content per result
- URLs are deduplicated across sections in full-paper generation

In [ ]:
from src.core.retrieval.web_search import search_topic_content

DEMO_TOPIC = "Knowledge Graphs for Healthcare Recommendation Systems"
DEMO_SECTION = "Introduction"
DEMO_KEYWORDS = "Knowledge Graph, Meta-path, LLM, Recommendation"

try:
    web_results = search_topic_content(DEMO_TOPIC, "", DEMO_SECTION)
    print(f"Retrieved {len(web_results)} web results")
    for i, r in enumerate(web_results[:3]):
        print(f"\nResult {i+1}: {r['title'][:80]}")
        print(f"  URL: {r['url'][:80]}")
        print(f"  Content: {r['content'][:120]}...")
except Exception as e:
    print(f"Web search unavailable: {e}")
    web_results = []

---
## Step 6: Prompt Construction

The prompt builder (`src/core/generation/prompts.py`) assembles a structured prompt containing:

- **Section-specific guidance** (e.g., "Summarize and synthesize related work" for Literature Review)
- **Dynamic word count targets** proportional to reference paper length
- **Structural variation instructions** (sentence diversity, burstiness, paragraph templates)
- **Style notes** from the analyzed reference papers
- **Keywords** to weave naturally into the text
- **Web search results** as grounding content
- **Image placeholder instructions** (Markdown syntax with search queries)
- **AI-tell ban list** (19 flagged words like 'delves', 'tapestry', 'paradigm shift')

In [ ]:
from src.core.generation.prompts import build_generation_prompt

prompt = build_generation_prompt(
    topic=DEMO_TOPIC,
    section=DEMO_SECTION,
    web_results=web_results,
    style_profile=style_profile,
    keywords=DEMO_KEYWORDS,
    include_visuals=True,
)

print(f"Prompt length: {len(prompt)} characters")
print(f"\n{'='*60}")
print(prompt[:800])
print(f"\n... ({len(prompt) - 800} more characters)")

---
## Step 7: LLM-Based Section Generation

The LLM Factory (`src/core/llm/llm_factory.py`) supports multiple providers via LangChain:

| Provider | Model | Library |
|----------|-------|---------|
| xAI | Grok-2 | `langchain_xai` |
| Groq | Various | `langchain_groq` |
| Google | Gemini 3.6 Flash | `langchain_google_genai` |
| OpenRouter | Any model | `langchain_openai` |

Generation includes retry logic (3 attempts) and a fallback that builds a minimal section from web snippets if the LLM fails completely.

In [ ]:
from src.core.llm.llm_factory import get_llm

print(f"Active LLM provider: {settings.llm_provider}")

try:
    llm = get_llm()
    response = llm.invoke(prompt)
    generated_text = response.content if response.content else ""
    print(f"\nGenerated section: {len(generated_text.split())} words")
    print(f"\nPreview:\n{generated_text[:400]}...")
except Exception as e:
    print(f"LLM unavailable: {e}")
    generated_text = (
        "Knowledge graphs have emerged as a useful tool for organizing and reasoning "
        "over structured biomedical data. In healthcare recommendation systems, these "
        "graphs capture relationships between diseases, drugs, symptoms, and treatments. "
        "Meta-path-based approaches traverse typed paths through the graph to discover "
        "meaningful patterns. Recent work combines LLM capabilities with graph-based "
        "retrieval to produce more accurate and explainable recommendations."
    )
    print(f"Using demo text ({len(generated_text.split())} words).")

---
## Step 8: Automated Evaluation & Iterative Refinement

The evaluator (`src/core/generation/evaluator.py`) runs **5 automated quality checks**:

1. **Fabricated citation detection** — flags `(Author, Year)` citations when no sources were retrieved
2. **Transition word overuse** — max 2 per paragraph (however, furthermore, moreover, etc.)
3. **Cross-section repetition** — 5-gram overlap detection against prior sections
4. **Style drift** — sentence length deviation > 50% from reference papers
5. **Length check** — proportional minimum based on reference paper word count

If evaluation fails, the section is **rewritten** using `build_rewrite_prompt()` which targets only the flagged issues (up to 3 attempts).

In [ ]:
from src.core.generation.evaluator import evaluate_section

evaluation = evaluate_section(
    text=generated_text,
    sources_used=[{"title": "demo", "url": "https://example.com"}],
    style_profile=style_profile,
    previous_sections_summary="",
    section_name=DEMO_SECTION,
)

print(f"Evaluation passed: {evaluation.passed}")
if evaluation.issues:
    print("Issues found:")
    for issue in evaluation.issues:
        print(f"  - {issue}")
else:
    print("No issues detected.")

print(f"\nRewrite loop: generate -> evaluate -> rewrite (if issues) -> re-evaluate")
print(f"Max attempts: 3")

---
## Step 9: Multi-Pass Humanization Pipeline

The **core innovation** of RSW. A 5-pass pipeline that removes AI writing patterns:

### Pass 1 — Pre-LLM Structural Work (pure Python, no API call)
- **1a. Burstiness Injection:** Randomises sentence lengths (target CV > 0.45)
- **1b. Sentence Restructurer:** Clause reordering, sentence-initial diversity
- **1c. Paragraph Mixer:** Paragraph-level structural variation

### Pass 2 — LLM Rewrite via Multi-Candidate Re-Ranking
- 3 candidates generated with different writing personas:
  1. **Empirical Researcher** (data-first voice)
  2. **Asymmetric Rhythm** (varied cadence)
  3. **Analytical Research Voice** (methodical tone)
- Each scored on burstiness CV, residual AI tells, composite score
- Best candidate selected

### Pass 3 — Post-LLM Processing (pure Python)
- **Lexical Booster:** Perplexity-raising word swaps (45% probability)
- **Lexical Diversity:** Synonym rotation for TTR improvement
- **Human Artifacts:** Hedges, parentheticals, cross-references, discourse markers

### Pass 4 — Quality Gate
- Composite score >= 80, Burstiness CV >= 0.45, Variety >= 0.35
- If below threshold, iterate Pass 2-3 again

### Pass 5 — Final Burstiness Enforcement

In [ ]:
from src.core.humanizer.burstiness import inject_burstiness, measure_burstiness, _split_into_sentences
from src.core.humanizer.sentence_restructurer import restructure_sentences, measure_sentence_initial_variety
from src.core.humanizer.paragraph_mixer import mix_paragraphs
from src.core.humanizer.lexical_booster import boost_perplexity, enforce_lexical_diversity
from src.core.humanizer.human_artifacts import inject_human_artifacts
from src.core.humanizer.reranker import score_candidate

# Pass 1a: Burstiness
cv_before = measure_burstiness(_split_into_sentences(generated_text))
text_p1a = inject_burstiness(generated_text)
cv_after = measure_burstiness(_split_into_sentences(text_p1a))
print(f"Pass 1a - Burstiness: CV {cv_before:.3f} -> {cv_after:.3f}  (target >= 0.45)")

# Pass 1b: Sentence restructuring
var_before = measure_sentence_initial_variety(text_p1a)
text_p1b = restructure_sentences(text_p1a)
var_after = measure_sentence_initial_variety(text_p1b)
print(f"Pass 1b - Variety:    {var_before:.3f} -> {var_after:.3f}  (target >= 0.35)")

# Pass 1c: Paragraph mixing
text_p1c = mix_paragraphs(text_p1b)
print(f"Pass 1c - Paragraph structure varied")

# Pass 3 (demo — post-LLM processing without LLM step)
text_p3 = boost_perplexity(text_p1c, swap_probability=0.45)
text_p3 = enforce_lexical_diversity(text_p3)
text_p3 = inject_human_artifacts(text_p3,
    hedge_probability=0.12, parenthetical_probability=0.18,
    cross_ref_probability=0.10, emphatic_probability=0.08,
    citation_connector_probability=0.12)
print(f"Pass 3  - Lexical boost + diversity + artifacts applied")

# Quality gate
score = score_candidate(text_p3)
variety = measure_sentence_initial_variety(text_p3)
print(f"\nQuality Gate:")
print(f"  Composite : {score['composite_score']:.1f}  (target >= 80)")
print(f"  Burstiness: {score['burstiness_cv']:.3f}  (target >= 0.45)")
print(f"  AI Tells  : {score['residual_tells']}      (target: 0)")
print(f"  Variety   : {variety:.3f}  (target >= 0.35)")

In [ ]:
# Full humanization pipeline (requires LLM access for Pass 2)
from src.core.humanizer.humanizer import humanize_section

demo_result = {
    "text": generated_text,
    "sources": [], "attempts": 1, "passed": True, "remaining_issues": [],
}

try:
    humanized = humanize_section(demo_result, style_notes=style_profile.to_prompt_text(), max_passes=2)
    m = humanized.get("humanization_metrics", {})
    print(f"Humanization complete (Deep - 2 passes):")
    print(f"  Burstiness CV : {m.get('burstiness_cv', 0):.3f}")
    print(f"  Residual tells: {m.get('residual_tells', 0)}")
    print(f"  Composite     : {m.get('composite_score', 0):.1f}")
    print(f"  Variety       : {m.get('sentence_initial_variety', 0):.3f}")
except Exception as e:
    print(f"Full humanization requires LLM: {e}")
    print("Pre-LLM and post-LLM steps above work without API access.")

---
## Step 10: Academic Image Search & Insertion

The image pipeline searches multiple sources for relevant academic figures:

1. **Tavily** — web image search
2. **Unsplash** — free academic/scientific photos
3. **Wikimedia Commons** — open diagrams and figures
4. **Europe PMC** — biomedical paper figures

**Deduplication:** Each image path is tracked in a `used_image_paths` set across sections.  
**Caching:** Cache key includes both topic AND keywords to prevent wrong reuse.

In [ ]:
from src.core.generation.generator import _build_image_query
from src.core.illustration.image_search import get_academic_illustration

query = _build_image_query(DEMO_TOPIC, "Methodology", DEMO_KEYWORDS)
print(f"Image search query: '{query}'")

try:
    img = get_academic_illustration(query)
    if img:
        print(f"Found: {img.get('source_name', '?')} -> {img.get('local_path', '?')}")
    else:
        print("No image found.")
except Exception as e:
    print(f"Image search unavailable: {e}")

---
## Step 11: PDF / DOCX Export

- **PDF:** Generated with `fpdf2` using DejaVu Sans fonts for Unicode support
- **DOCX:** Generated with `python-docx` with proper heading/paragraph styles

In [ ]:
from src.core.export.pdf_export import generate_pdf
from src.core.export.docx_export import generate_docx

demo_sections = [
    ("Introduction", generated_text),
    ("Methodology", "This section describes the methodology."),
]

pdf_bytes = generate_pdf(DEMO_TOPIC, demo_sections)
docx_bytes = generate_docx(DEMO_TOPIC, demo_sections)

print(f"PDF : {len(pdf_bytes):,} bytes")
print(f"DOCX: {len(docx_bytes):,} bytes")

# Save to deliverables folder
with open("demo_output.pdf", "wb") as f:
    f.write(pdf_bytes)
with open("demo_output.docx", "wb") as f:
    f.write(docx_bytes)
print("\nFiles saved: demo_output.pdf, demo_output.docx")

---
## Step 12: Full End-to-End Paper Generation

The `generate_full_paper()` function is a Python **generator** that yields `(section_name, result)` for each section sequentially:

1. Searches web per section (deduplicating URLs)
2. Generates, evaluates, and rewrites each section
3. Maintains running summaries for cross-section coherence
4. Manages image cache and deduplication
5. Builds APA-style references from all sources used

The Streamlit app wraps this with `humanize_section()` per section and renders results live.

In [ ]:
from src.core.generation.generator import generate_full_paper, DEFAULT_SECTIONS

print("Sections generated in order:")
for s in DEFAULT_SECTIONS:
    print(f"  - {s}")

print(f"\nUsage in Streamlit app:")
print("""
for section_name, result in generate_full_paper(
    topic, style_profile, keywords=keywords, include_visuals=True
):
    # Humanize (skip References)
    if section_name.lower() != 'references':
        result = humanize_section(result, style_notes, max_passes=2)
    
    # Display section
    st.subheader(section_name)
    render_section_content(result['text'])
    
    # Collect for export
    sections.append((section_name, result['text']))

# Export
pdf_bytes = generate_pdf(topic, sections)
docx_bytes = generate_docx(topic, sections)
""")

print("\nTo run the Streamlit app:")
print("  streamlit run apps/app_generation.py")

---
## Summary

The RSW pipeline transforms a topic + keywords + reference papers into a complete, humanized academic research paper through:

1. **Style learning** from reference papers (12 quantitative features)
2. **Grounded generation** using live web search results
3. **Automated quality checks** with iterative refinement
4. **Multi-pass humanization** targeting AI detection patterns
5. **Academic image insertion** from multiple sources
6. **Professional export** as PDF or DOCX

The key innovation is the humanization pipeline which combines structural text manipulation (burstiness, clause reordering, lexical substitution) with LLM-based multi-candidate re-ranking to produce text that reads like authentic human scholarly prose.